# Retail Sales Performance Analysis — Data Preparation

## Objective

This notebook converts the raw Online Retail II dataset into a clean, reproducible, analysis-ready dataset using the data-quality findings and business rules established during exploratory profiling.

The processed dataset will serve as the common analytical source for SQL analysis, Excel reporting, and Tableau dashboard development.

### Preparation Workflow

1. Load and standardize the raw annual worksheets
2. Resolve overlapping records between source worksheets
3. Remove confirmed exact duplicate transaction lines
4. Apply missing-value treatment decisions
5. Classify sales, returns, cancellations, and operational adjustments
6. Separate merchandise transactions from non-product activity
7. Standardize geographic fields
8. Create analytical date and business-metric fields
9. Validate the processed dataset
10. Export analysis-ready datasets

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

In [3]:
raw_file = Path("../data/raw/online_retail_II.xlsx")
processed_dir = Path("../data/processed")

processed_dir.mkdir(parents=True, exist_ok=True)

print("Raw file exists:", raw_file.exists())
print("Processed directory exists:", processed_dir.exists())

Raw file exists: True
Processed directory exists: True


## 1. Raw Data Ingestion

Both annual worksheets are loaded independently. Identifier fields are treated as strings to preserve their categorical meaning and avoid unintended numerical interpretation.

In [4]:
identifier_types = {
    "Invoice": "string",
    "StockCode": "string",
    "Description": "string",
    "Customer ID": "string",
    "Country": "string"
}

In [5]:
sales_2009_2010 = pd.read_excel(
    raw_file,
    sheet_name="Year 2009-2010",
    dtype=identifier_types
)

sales_2009_2010["SourcePeriod"] = "2009-2010"

In [6]:
sales_2010_2011 = pd.read_excel(
    raw_file,
    sheet_name="Year 2010-2011",
    dtype=identifier_types
)

sales_2010_2011["SourcePeriod"] = "2010-2011"

In [7]:
print("2009-2010:", sales_2009_2010.shape)
print("2010-2011:", sales_2010_2011.shape)

print(
    "Columns identical:",
    sales_2009_2010.columns.tolist()
    == sales_2010_2011.columns.tolist()
)

2009-2010: (525461, 9)
2010-2011: (541910, 9)
Columns identical: True


## 2. Source Worksheet Overlap Resolution

The two annual worksheets contain an overlapping transaction period beginning on 1 December 2010.

Exploratory profiling confirmed that transactions in this overlapping period are reproduced across both source worksheets. To prevent double-counting while preserving the most recent source version, overlapping records are retained from the 2010–2011 worksheet and removed from the 2009–2010 worksheet.

This source-level overlap is resolved before ordinary duplicate transaction records are addressed.

In [8]:
overlap_start = pd.Timestamp("2010-12-01")

overlap_2009_2010 = sales_2009_2010[
    sales_2009_2010["InvoiceDate"] >= overlap_start
].copy()

overlap_2010_2011 = sales_2010_2011[
    sales_2010_2011["InvoiceDate"] < pd.Timestamp("2010-12-10")
].copy()

print("Overlap rows in 2009-2010:", len(overlap_2009_2010))
print("Overlap rows in 2010-2011:", len(overlap_2010_2011))

print(
    "2009-2010 overlap range:",
    overlap_2009_2010["InvoiceDate"].min(),
    "to",
    overlap_2009_2010["InvoiceDate"].max()
)

print(
    "2010-2011 overlap range:",
    overlap_2010_2011["InvoiceDate"].min(),
    "to",
    overlap_2010_2011["InvoiceDate"].max()
)

Overlap rows in 2009-2010: 22523
Overlap rows in 2010-2011: 22523
2009-2010 overlap range: 2010-12-01 08:26:00 to 2010-12-09 20:01:00
2010-2011 overlap range: 2010-12-01 08:26:00 to 2010-12-09 20:01:00


In [9]:
sales_2009_2010_trimmed = sales_2009_2010[
    sales_2009_2010["InvoiceDate"] < overlap_start
].copy()

In [10]:
print("Original 2009-2010 rows:", len(sales_2009_2010))
print("Rows retained:", len(sales_2009_2010_trimmed))
print(
    "Rows removed as source overlap:",
    len(sales_2009_2010) - len(sales_2009_2010_trimmed)
)

Original 2009-2010 rows: 525461
Rows retained: 502938
Rows removed as source overlap: 22523


In [11]:
sales_combined = pd.concat(
    [
        sales_2009_2010_trimmed,
        sales_2010_2011
    ],
    ignore_index=True
)

In [12]:
original_combined_rows = (
    len(sales_2009_2010)
    + len(sales_2010_2011)
)

print("Original combined rows:", original_combined_rows)
print("Rows after overlap resolution:", len(sales_combined))
print(
    "Rows removed due to source overlap:",
    original_combined_rows - len(sales_combined)
)

Original combined rows: 1067371
Rows after overlap resolution: 1044848
Rows removed due to source overlap: 22523


## 3. Exact Duplicate Removal

After resolving the annual worksheet overlap, exact duplicate transaction-line records are removed from the retained dataset.

Duplicates are evaluated using the transaction-level fields that describe a line item. The first occurrence is retained to avoid double-counting sales, quantities, and returns while preserving one valid representation of each transaction line.

In [13]:
duplicate_key_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]

In [14]:
duplicate_count = sales_combined.duplicated(
    subset=duplicate_key_columns
).sum()

print("Exact duplicate occurrences:", duplicate_count)

Exact duplicate occurrences: 11812


In [15]:
sales_dedup = (
    sales_combined
    .drop_duplicates(
        subset=duplicate_key_columns,
        keep="first"
    )
    .copy()
)

In [16]:
print("Rows before duplicate removal:", len(sales_combined))
print("Duplicate occurrences removed:", len(sales_combined) - len(sales_dedup))
print("Rows after duplicate removal:", len(sales_dedup))

Rows before duplicate removal: 1044848
Duplicate occurrences removed: 11812
Rows after duplicate removal: 1033036


In [17]:
remaining_duplicates = sales_dedup.duplicated(
    subset=duplicate_key_columns
).sum()

print("Remaining exact duplicates:", remaining_duplicates)

Remaining exact duplicates: 0


## 4. Missing Value Treatment

Missing values are treated according to their analytical impact rather than removed indiscriminately.

### Missing Product Description

Profiling identified 4,275 records with missing product descriptions. These records also had zero unit price and were consistent with operational or inventory-adjustment activity rather than normal merchandise sales. They are excluded from the analytical dataset.

### Missing Customer ID

Transactions with missing Customer ID are retained because many represent valid revenue-generating transactions. These records remain available for sales, product, geographic, and time-based analysis but are excluded from analyses that require customer identity.

A customer-identification flag is created to distinguish known and unidentified customers.

In [18]:
print(
    "Missing Descriptions:",
    sales_dedup["Description"].isna().sum()
)

print(
    "Missing Customer IDs:",
    sales_dedup["Customer ID"].isna().sum()
)

Missing Descriptions: 4275
Missing Customer IDs: 235151


In [19]:
sales_working = sales_dedup[
    sales_dedup["Description"].notna()
].copy()

In [20]:
print("Rows before missing-description removal:", len(sales_dedup))

print(
    "Missing-description rows removed:",
    len(sales_dedup) - len(sales_working)
)

print("Working dataset rows:", len(sales_working))

Rows before missing-description removal: 1033036
Missing-description rows removed: 4275
Working dataset rows: 1028761


In [21]:
sales_working["CustomerKnown"] = (
    sales_working["Customer ID"].notna()
)

In [22]:
print(
    "Known-customer rows:",
    sales_working["CustomerKnown"].sum()
)

print(
    "Unidentified-customer rows:",
    (~sales_working["CustomerKnown"]).sum()
)

print(
    "Remaining missing descriptions:",
    sales_working["Description"].isna().sum()
)

Known-customer rows: 797885
Unidentified-customer rows: 230876
Remaining missing descriptions: 0


In [23]:
sales_working["CustomerKnown"].value_counts()

CustomerKnown
True     797885
False    230876
Name: count, dtype: int64

## 5. Transaction Classification

Transaction records are classified using invoice-number and quantity behavior identified during exploratory profiling.

Cancellation-style invoices beginning with "C" and containing negative quantities are treated as cancellations or returns.

Negative-quantity records without a cancellation-style invoice are classified separately as operational adjustments when they carry no transaction price.

Rare cancellation-style invoices with non-negative quantity are retained as manual or exceptional adjustments for transparency rather than being treated as normal merchandise sales.

In [24]:
sales_working["IsCancellationInvoice"] = (
    sales_working["Invoice"]
    .str.startswith("C", na=False)
)

In [25]:
print(
    "Negative quantity rows:",
    (sales_working["Quantity"] < 0).sum()
)

print(
    "Cancellation invoice rows:",
    sales_working["IsCancellationInvoice"].sum()
)

Negative quantity rows: 19863
Cancellation invoice rows: 19104


In [26]:
pd.crosstab(
    sales_working["IsCancellationInvoice"],
    sales_working["Quantity"] < 0,
    rownames=["Cancellation Invoice"],
    colnames=["Negative Quantity"]
)

Negative Quantity,False,True
Cancellation Invoice,,
False,1008897,760
True,1,19103


In [27]:
sales_working["TransactionType"] = "Sale"

In [28]:
sales_working.loc[
    sales_working["IsCancellationInvoice"]
    & (sales_working["Quantity"] < 0),
    "TransactionType"
] = "Cancellation/Return"

In [29]:
sales_working.loc[
    (~sales_working["IsCancellationInvoice"])
    & (sales_working["Quantity"] < 0)
    & (sales_working["Price"] == 0),
    "TransactionType"
] = "Operational Adjustment"

In [30]:
sales_working.loc[
    sales_working["IsCancellationInvoice"]
    & (sales_working["Quantity"] >= 0),
    "TransactionType"
] = "Manual/Exceptional Adjustment"

In [31]:
sales_working["TransactionType"].value_counts()

TransactionType
Sale                             1008897
Cancellation/Return                19103
Operational Adjustment               760
Manual/Exceptional Adjustment          1
Name: count, dtype: int64

In [32]:
print(
    "Classified rows:",
    sales_working["TransactionType"].value_counts().sum()
)

print(
    "Working dataset rows:",
    len(sales_working)
)

Classified rows: 1028761
Working dataset rows: 1028761


### Zero-Price / Non-Revenue Transactions

Positive-quantity records initially classified as sales but carrying a zero unit price are separated from revenue-generating sales.

These records may represent free items, samples, testing records, manual activity, or other non-revenue transactions. They are retained for transparency but excluded from merchandise revenue calculations.

In [33]:
zero_price_mask = (
    (sales_working["TransactionType"] == "Sale")
    & (sales_working["Price"] == 0)
)

print(
    "Zero-price sale rows:",
    zero_price_mask.sum()
)

Zero-price sale rows: 979


In [34]:
print(
    "Minimum quantity:",
    sales_working.loc[zero_price_mask, "Quantity"].min()
)

print(
    "Maximum quantity:",
    sales_working.loc[zero_price_mask, "Quantity"].max()
)

print(
    "Total zero-price units:",
    sales_working.loc[zero_price_mask, "Quantity"].sum()
)

Minimum quantity: 1
Maximum quantity: 12540
Total zero-price units: 69421


In [35]:
sales_working.loc[
    zero_price_mask,
    "TransactionType"
] = "Zero-Price/Non-Revenue"

In [36]:
sales_working["TransactionType"].value_counts()

TransactionType
Sale                             1007918
Cancellation/Return                19103
Zero-Price/Non-Revenue               979
Operational Adjustment               760
Manual/Exceptional Adjustment          1
Name: count, dtype: int64

In [37]:
print(
    "Total classified rows:",
    sales_working["TransactionType"].value_counts().sum()
)

print(
    "Dataset rows:",
    len(sales_working)
)

Total classified rows: 1028761
Dataset rows: 1028761


## 6. Merchandise and Non-Product Line Classification

Transaction type and product-line type represent separate analytical dimensions.

Some positive-priced transaction lines represent postage, discounts, commissions, manual adjustments, samples, or test activity rather than merchandise products. These records are retained in the processed dataset but classified separately so merchandise KPIs and product-performance analysis are not distorted by non-product activity.

The classification rules below reproduce the categories validated during exploratory profiling.

In [38]:
sales_working["LineCategory"] = "Merchandise"

In [39]:
shipping_codes = [
    "POST",
    "DOT",
    "C2"
]

sales_working.loc[
    sales_working["StockCode"].isin(shipping_codes),
    "LineCategory"
] = "Shipping/Postage"

In [40]:
manual_financial_codes = [
    "M",
    "m",
    "B",
    "BANK CHARGES"
]

sales_working.loc[
    sales_working["StockCode"].isin(manual_financial_codes),
    "LineCategory"
] = "Manual/Financial Adjustment"

In [41]:
sales_working.loc[
    sales_working["StockCode"] == "D",
    "LineCategory"
] = "Discount"

In [42]:
sales_working.loc[
    sales_working["StockCode"] == "S",
    "LineCategory"
] = "Sample"

In [43]:
operational_codes = [
    "ADJUST",
    "ADJUST2"
]

sales_working.loc[
    sales_working["StockCode"].isin(operational_codes),
    "LineCategory"
] = "Operational Adjustment"

In [44]:
marketplace_fee_codes = [
    "AMAZONFEE",
    "CRUK"
]

sales_working.loc[
    sales_working["StockCode"].isin(marketplace_fee_codes),
    "LineCategory"
] = "Marketplace/Commission Fee"

In [45]:
test_codes = [
    "TEST001",
    "TEST002"
]

sales_working.loc[
    sales_working["StockCode"].isin(test_codes),
    "LineCategory"
] = "Test Record"

In [46]:
sales_working["LineCategory"].value_counts()

LineCategory
Merchandise                    1023075
Shipping/Postage                  3776
Manual/Financial Adjustment       1498
Discount                           173
Sample                             101
Operational Adjustment              70
Marketplace/Commission Fee          52
Test Record                         16
Name: count, dtype: int64

In [47]:
print(
    "Line-category rows:",
    sales_working["LineCategory"].value_counts().sum()
)

print(
    "Dataset rows:",
    len(sales_working)
)

Line-category rows: 1028761
Dataset rows: 1028761


In [48]:
pd.crosstab(
    sales_working["LineCategory"],
    sales_working["TransactionType"]
)

TransactionType,Cancellation/Return,Manual/Exceptional Adjustment,Operational Adjustment,Sale,Zero-Price/Non-Revenue
LineCategory,,,,,
Discount,168,0,0,5,0
Manual/Financial Adjustment,599,1,0,891,7
Marketplace/Commission Fee,49,0,0,3,0
Merchandise,17916,0,760,1003434,965
Operational Adjustment,31,0,0,39,0
Sample,98,0,0,3,0
Shipping/Postage,238,0,0,3533,5
Test Record,4,0,0,10,2


## 7. Business Metric Engineering

Analytical measures are derived from the cleaned transaction classifications.

Merchandise sales and returns are measured separately so gross sales, return value, net revenue, sold units, and returned units can be analyzed independently.

Zero-price merchandise activity is retained but excluded from paid merchandise sales and units-sold KPIs.

In [49]:
sales_working["LineValue"] = (
    sales_working["Quantity"]
    * sales_working["Price"]
)

In [50]:
sales_working["GrossSales"] = np.where(
    (sales_working["LineCategory"] == "Merchandise")
    & (sales_working["TransactionType"] == "Sale"),
    sales_working["LineValue"],
    0
)

In [51]:
sales_working["ReturnValue"] = np.where(
    (sales_working["LineCategory"] == "Merchandise")
    & (sales_working["TransactionType"] == "Cancellation/Return"),
    -sales_working["LineValue"],
    0
)

In [52]:
sales_working["NetMerchandiseRevenue"] = (
    sales_working["GrossSales"]
    - sales_working["ReturnValue"]
)

In [68]:
sales_working["UnitsSold"] = np.where(
    (sales_working["LineCategory"] == "Merchandise")
    & (sales_working["TransactionType"] == "Sale")
    & (sales_working["Price"] > 0),
    sales_working["Quantity"],
    0
)

In [69]:
sales_working["ReturnedUnits"] = np.where(
    (sales_working["LineCategory"] == "Merchandise")
    & (sales_working["TransactionType"] == "Cancellation/Return"),
    -sales_working["Quantity"],
    0
)

In [70]:
sales_working["ZeroPriceUnits"] = np.where(
    (sales_working["LineCategory"] == "Merchandise")
    & (sales_working["TransactionType"] == "Zero-Price/Non-Revenue"),
    sales_working["Quantity"],
    0
)

In [71]:
sales_working["NetUnits"] = (
    sales_working["UnitsSold"]
    - sales_working["ReturnedUnits"]
)

In [72]:
gross_sales = sales_working["GrossSales"].sum()
return_value = sales_working["ReturnValue"].sum()
net_revenue = sales_working["NetMerchandiseRevenue"].sum()

units_sold = sales_working["UnitsSold"].sum()
returned_units = sales_working["ReturnedUnits"].sum()
zero_price_units = sales_working["ZeroPriceUnits"].sum()
net_units = sales_working["NetUnits"].sum()

print("Paid Units Sold:", units_sold)
print("Returned Units:", returned_units)
print("Zero-Price Merchandise Units:", zero_price_units)
print("Net Paid Units:", net_units)

Paid Units Sold: 11188141
Returned Units: 467741
Zero-Price Merchandise Units: 69399
Net Paid Units: 10720400


In [73]:
print(
    "Gross Sales - Return Value:",
    round(gross_sales - return_value, 2)
)

print(
    "Calculated Net Revenue:",
    round(net_revenue, 2)
)

print(
    "Revenue reconciliation:",
    round(gross_sales - return_value, 2)
    == round(net_revenue, 2)
)

Gross Sales - Return Value: 18929085.68
Calculated Net Revenue: 18929085.68
Revenue reconciliation: True


In [74]:
positive_merchandise_units = sales_working.loc[
    (sales_working["LineCategory"] == "Merchandise")
    & (sales_working["Quantity"] > 0),
    "Quantity"
].sum()

print(
    "Paid + Zero-Price Units:",
    units_sold + zero_price_units
)

print(
    "All Positive Merchandise Units:",
    positive_merchandise_units
)

print(
    "Unit reconciliation:",
    units_sold + zero_price_units
    == positive_merchandise_units
)

Paid + Zero-Price Units: 11257540
All Positive Merchandise Units: 11257540
Unit reconciliation: True


## 8. Date and Time Feature Engineering

Date and time attributes are derived from InvoiceDate to support trend, seasonality, period-over-period, and dashboard analysis.

The original InvoiceDate is preserved while additional analytical fields are created for year, quarter, month, year-month, weekday, date, and hour.

In [75]:
# Ensure InvoiceDate is datetime
sales_working["InvoiceDate"] = pd.to_datetime(
    sales_working["InvoiceDate"]
)

In [76]:
sales_working["Year"] = sales_working["InvoiceDate"].dt.year

sales_working["Quarter"] = (
    "Q" + sales_working["InvoiceDate"].dt.quarter.astype(str)
)

sales_working["MonthNumber"] = (
    sales_working["InvoiceDate"].dt.month
)

sales_working["Month"] = (
    sales_working["InvoiceDate"].dt.month_name()
)

sales_working["YearMonth"] = (
    sales_working["InvoiceDate"]
    .dt.to_period("M")
    .astype(str)
)

sales_working["Weekday"] = (
    sales_working["InvoiceDate"].dt.day_name()
)

sales_working["Hour"] = (
    sales_working["InvoiceDate"].dt.hour
)

sales_working["InvoiceDateOnly"] = (
    sales_working["InvoiceDate"].dt.date
)

In [77]:
sales_working[
    [
        "InvoiceDate",
        "Year",
        "Quarter",
        "MonthNumber",
        "Month",
        "YearMonth",
        "Weekday",
        "Hour",
        "InvoiceDateOnly"
    ]
].head(10)

,InvoiceDate,Year,Quarter,MonthNumber,Month,YearMonth,Weekday,Hour,InvoiceDateOnly
0,2009-12-01 07:45:00,2009,Q4,12,December,2009-12,Tuesday,7,2009-12-01
1,2009-12-01 07:45:00,2009,Q4,12,December,2009-12,Tuesday,7,2009-12-01
2,2009-12-01 07:45:00,2009,Q4,12,December,2009-12,Tuesday,7,2009-12-01
3,2009-12-01 07:45:00,2009,Q4,12,December,2009-12,Tuesday,7,2009-12-01
4,2009-12-01 07:45:00,2009,Q4,12,December,2009-12,Tuesday,7,2009-12-01
5,2009-12-01 07:45:00,2009,Q4,12,December,2009-12,Tuesday,7,2009-12-01
6,2009-12-01 07:45:00,2009,Q4,12,December,2009-12,Tuesday,7,2009-12-01
7,2009-12-01 07:45:00,2009,Q4,12,December,2009-12,Tuesday,7,2009-12-01
8,2009-12-01 07:46:00,2009,Q4,12,December,2009-12,Tuesday,7,2009-12-01
9,2009-12-01 07:46:00,2009,Q4,12,December,2009-12,Tuesday,7,2009-12-01


In [78]:
print(
    "Earliest transaction:",
    sales_working["InvoiceDate"].min()
)

print(
    "Latest transaction:",
    sales_working["InvoiceDate"].max()
)

print(
    "Years:",
    sorted(sales_working["Year"].unique())
)

Earliest transaction: 2009-12-01 07:45:00
Latest transaction: 2011-12-09 12:50:00
Years: [np.int32(2009), np.int32(2010), np.int32(2011)]


In [79]:
print(
    "Number of YearMonth periods:",
    sales_working["YearMonth"].nunique()
)

print(
    sorted(
        sales_working["YearMonth"].unique()
    )[:5]
)

print(
    sorted(
        sales_working["YearMonth"].unique()
    )[-5:]
)

Number of YearMonth periods: 25
['2009-12', '2010-01', '2010-02', '2010-03', '2010-04']
['2011-08', '2011-09', '2011-10', '2011-11', '2011-12']


In [80]:
sales_working.groupby("Year").agg(
    Rows=("Invoice", "size"),
    Orders=("Invoice", "nunique")
)

,Rows,Orders
Year,,
2009,44494,2102
2010,490935,24705
2011,493332,22546


In [81]:
sales_working["WeekdayNumber"] = (
    sales_working["InvoiceDate"].dt.dayofweek + 1
)

In [82]:
sales_working[
    ["Weekday", "WeekdayNumber"]
].drop_duplicates().sort_values(
    "WeekdayNumber"
)

,Weekday,WeekdayNumber
14405,Monday,1
0,Tuesday,2
3223,Wednesday,3
6500,Thursday,4
9502,Friday,5
12061,Saturday,6
12463,Sunday,7


## 9. Geographic Standardization

Country values are standardized while preserving the original geographic field.

Alternative country labels are mapped to consistent names to improve grouping and geographic recognition in downstream visualization tools.

Non-specific geographic labels are retained in the dataset but flagged separately so they can be excluded from map-based analysis without removing their transaction value from overall business metrics.

In [83]:
print(
    "Missing Country values:",
    sales_working["Country"].isna().sum()
)

print(
    "Unique raw country values:",
    sales_working["Country"].nunique()
)

Missing Country values: 0
Unique raw country values: 43


In [84]:
sales_working["CountryStandardized"] = (
    sales_working["Country"]
    .str.strip()
)

In [85]:
country_mapping = {
    "EIRE": "Ireland",
    "USA": "United States",
    "Korea": "South Korea"
}

sales_working["CountryStandardized"] = (
    sales_working["CountryStandardized"]
    .replace(country_mapping)
)

In [86]:
sorted(
    sales_working["CountryStandardized"]
    .dropna()
    .unique()
)

['Australia',
 'Austria',
 'Bahrain',
 'Belgium',
 'Bermuda',
 'Brazil',
 'Canada',
 'Channel Islands',
 'Cyprus',
 'Czech Republic',
 'Denmark',
 'European Community',
 'Finland',
 'France',
 'Germany',
 'Greece',
 'Hong Kong',
 'Iceland',
 'Ireland',
 'Israel',
 'Italy',
 'Japan',
 'Lebanon',
 'Lithuania',
 'Malta',
 'Netherlands',
 'Nigeria',
 'Norway',
 'Poland',
 'Portugal',
 'RSA',
 'Saudi Arabia',
 'Singapore',
 'South Korea',
 'Spain',
 'Sweden',
 'Switzerland',
 'Thailand',
 'United Arab Emirates',
 'United Kingdom',
 'United States',
 'Unspecified',
 'West Indies']

In [87]:
non_specific_geographies = [
    "Unspecified",
    "European Community",
    "West Indies"
]

sales_working["MapEligible"] = (
    ~sales_working["CountryStandardized"]
    .isin(non_specific_geographies)
)

In [88]:
sales_working["MapEligible"].value_counts()

MapEligible
True     1027894
False        867
Name: count, dtype: int64

In [89]:
print(
    "Raw country values:",
    sales_working["Country"].nunique()
)

print(
    "Standardized country values:",
    sales_working["CountryStandardized"].nunique()
)

print(
    "Map-eligible rows:",
    sales_working["MapEligible"].sum()
)

print(
    "Non-map-eligible rows:",
    (~sales_working["MapEligible"]).sum()
)

Raw country values: 43
Standardized country values: 43
Map-eligible rows: 1027894
Non-map-eligible rows: 867


In [90]:
sales_working[
    sales_working["Country"]
    != sales_working["CountryStandardized"]
][
    ["Country", "CountryStandardized"]
].drop_duplicates()

,Country,CountryStandardized
126,USA,United States
440,EIRE,Ireland
339885,Korea,South Korea


In [91]:
country_validation = (
    sales_working[
        sales_working["MapEligible"]
    ]
    .groupby("CountryStandardized")
    .agg(
        GrossSales=("GrossSales", "sum"),
        ReturnValue=("ReturnValue", "sum"),
        NetRevenue=("NetMerchandiseRevenue", "sum"),
        Orders=("Invoice", "nunique")
    )
    .reset_index()
    .sort_values(
        "NetRevenue",
        ascending=False
    )
)

country_validation.head(15)

,CountryStandardized,GrossSales,ReturnValue,NetRevenue,Orders
38,United Kingdom,1.680278e+07,632183.00,1.617059e+07,44833
17,Ireland,6.234142e+05,20297.55,6.031166e+05,806
24,Netherlands,5.497734e+05,3677.98,5.460954e+05,250
13,Germany,3.832890e+05,8715.49,3.745735e+05,1095
12,France,3.110903e+05,17659.09,2.934312e+05,746
0,Australia,1.678000e+05,1517.86,1.662821e+05,117
35,Switzerland,9.402459e+04,1106.53,9.291806e+04,123
33,Spain,9.776675e+04,12955.19,8.481156e+04,188
34,Sweden,8.631914e+04,1973.97,8.434517e+04,129
10,Denmark,6.742269e+04,4067.10,6.335559e+04,53


## 10. Analytical Flags and Final Dataset Structure

Additional analytical flags are created to simplify downstream filtering and KPI calculations.

The flags distinguish revenue-generating merchandise, returns, identified customers, and geographically mappable transactions without removing the underlying transaction records.

These fields allow downstream SQL, Excel, and Tableau analyses to apply consistent business definitions.

In [92]:
sales_working["IsPaidMerchandiseSale"] = (
    (sales_working["LineCategory"] == "Merchandise")
    & (sales_working["TransactionType"] == "Sale")
    & (sales_working["Price"] > 0)
)

In [93]:
print(
    "Paid merchandise sale rows:",
    sales_working["IsPaidMerchandiseSale"].sum()
)

Paid merchandise sale rows: 1003434


In [94]:
sales_working["IsMerchandiseReturn"] = (
    (sales_working["LineCategory"] == "Merchandise")
    & (sales_working["TransactionType"] == "Cancellation/Return")
)

In [95]:
print(
    "Merchandise return rows:",
    sales_working["IsMerchandiseReturn"].sum()
)

Merchandise return rows: 17916


In [96]:
sales_working["IsRevenueGenerating"] = (
    sales_working["GrossSales"] > 0
)

In [97]:
sales_working["CustomerKnown"].value_counts()

CustomerKnown
True     797885
False    230876
Name: count, dtype: int64

In [98]:
flag_summary = pd.DataFrame({
    "Flag": [
        "Paid Merchandise Sale",
        "Merchandise Return",
        "Revenue Generating",
        "Known Customer",
        "Map Eligible"
    ],
    "Rows": [
        sales_working["IsPaidMerchandiseSale"].sum(),
        sales_working["IsMerchandiseReturn"].sum(),
        sales_working["IsRevenueGenerating"].sum(),
        sales_working["CustomerKnown"].sum(),
        sales_working["MapEligible"].sum()
    ]
})

flag_summary

,Flag,Rows
0,Paid Merchandise Sale,1003434
1,Merchandise Return,17916
2,Revenue Generating,1003434
3,Known Customer,797885
4,Map Eligible,1027894


In [99]:
print(
    "Final column count:",
    len(sales_working.columns)
)

print()

print(
    sales_working.columns.tolist()
)

Final column count: 35

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'SourcePeriod', 'CustomerKnown', 'IsCancellationInvoice', 'TransactionType', 'LineCategory', 'LineValue', 'GrossSales', 'ReturnValue', 'NetMerchandiseRevenue', 'UnitsSold', 'ReturnedUnits', 'ZeroPriceUnits', 'NetUnits', 'Year', 'Quarter', 'MonthNumber', 'Month', 'YearMonth', 'Weekday', 'Hour', 'InvoiceDateOnly', 'WeekdayNumber', 'CountryStandardized', 'MapEligible', 'IsPaidMerchandiseSale', 'IsMerchandiseReturn', 'IsRevenueGenerating']


In [101]:
key_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]

In [102]:
print("Rows:", len(sales_working))
print("Columns:", len(sales_working.columns))

print(
    "Duplicate transaction lines:",
    sales_working.duplicated(
        subset=key_columns
    ).sum()
)

print(
    "Missing descriptions:",
    sales_working["Description"].isna().sum()
)

print(
    "Missing InvoiceDate:",
    sales_working["InvoiceDate"].isna().sum()
)

print(
    "Missing Country:",
    sales_working["Country"].isna().sum()
)

Rows: 1028761
Columns: 35
Duplicate transaction lines: 0
Missing descriptions: 0
Missing InvoiceDate: 0
Missing Country: 0


In [103]:
print(
    "GrossSales on non-paid rows:",
    sales_working.loc[
        ~sales_working["IsPaidMerchandiseSale"],
        "GrossSales"
    ].sum()
)

print(
    "ReturnValue on non-return rows:",
    sales_working.loc[
        ~sales_working["IsMerchandiseReturn"],
        "ReturnValue"
    ].sum()
)

GrossSales on non-paid rows: 0.0
ReturnValue on non-return rows: 0.0


## 11. Export Analysis-Ready Dataset

The validated transaction-level dataset is exported to the processed-data directory for downstream SQL, Excel, and Tableau analysis.

The exported dataset preserves the cleaned source fields, transaction classifications, engineered business metrics, temporal attributes, geographic standardization, and analytical flags created during the preparation workflow.

In [105]:
final_columns = [
    # Original / source fields
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
    "SourcePeriod",

    # Classification fields
    "CustomerKnown",
    "IsCancellationInvoice",
    "TransactionType",
    "LineCategory",

    # Business metric fields
    "LineValue",
    "GrossSales",
    "ReturnValue",
    "NetMerchandiseRevenue",
    "UnitsSold",
    "ReturnedUnits",
    "ZeroPriceUnits",
    "NetUnits",

    # Date / time fields
    "Year",
    "Quarter",
    "MonthNumber",
    "Month",
    "YearMonth",
    "Weekday",
    "WeekdayNumber",
    "Hour",
    "InvoiceDateOnly",

    # Geography
    "CountryStandardized",
    "MapEligible",

    # Analytical flags
    "IsPaidMerchandiseSale",
    "IsMerchandiseReturn",
    "IsRevenueGenerating"
]

In [106]:
sales_final = sales_working[final_columns].copy()

print("Final rows:", len(sales_final))
print("Final columns:", len(sales_final.columns))

Final rows: 1028761
Final columns: 35


In [107]:
processed_file = (
    processed_dir
    / "retail_sales_analysis_ready.csv"
)

sales_final.to_csv(
    processed_file,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print("Export complete.")
print("File:", processed_file)
print("File exists:", processed_file.exists())

Export complete.
File: ../data/processed/retail_sales_analysis_ready.csv
File exists: True


In [108]:
file_size_mb = (
    processed_file.stat().st_size
    / (1024 ** 2)
)

print(
    "Processed file size:",
    round(file_size_mb, 2),
    "MB"
)

Processed file size: 241.02 MB


In [111]:
export_test = pd.read_csv(
    processed_file,
    nrows=5
)

export_test

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourcePeriod,CustomerKnown,...,YearMonth,Weekday,WeekdayNumber,Hour,InvoiceDateOnly,CountryStandardized,MapEligible,IsPaidMerchandiseSale,IsMerchandiseReturn,IsRevenueGenerating
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,2009-2010,True,...,2009-12,Tuesday,2,7,2009-12-01,United Kingdom,True,True,False,True
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010,True,...,2009-12,Tuesday,2,7,2009-12-01,United Kingdom,True,True,False,True
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009-2010,True,...,2009-12,Tuesday,2,7,2009-12-01,United Kingdom,True,True,False,True
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,2009-2010,True,...,2009-12,Tuesday,2,7,2009-12-01,United Kingdom,True,True,False,True
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,2009-2010,True,...,2009-12,Tuesday,2,7,2009-12-01,United Kingdom,True,True,False,True


In [112]:
print(
    "Exported column count:",
    len(export_test.columns)
)

print(
    "Columns match:",
    export_test.columns.tolist()
    == sales_final.columns.tolist()
)

Exported column count: 35
Columns match: True


In [110]:
print("=== DATA PREPARATION FINAL CHECK ===")

print("Rows:", len(sales_final))
print("Columns:", len(sales_final.columns))
print("Exact duplicates:", sales_final.duplicated(subset=key_columns).sum())
print("Missing descriptions:", sales_final["Description"].isna().sum())
print("Missing dates:", sales_final["InvoiceDate"].isna().sum())
print("Missing countries:", sales_final["Country"].isna().sum())

print()
print("Gross Merchandise Sales:", round(sales_final["GrossSales"].sum(), 2))
print("Return Value:", round(sales_final["ReturnValue"].sum(), 2))
print(
    "Net Merchandise Revenue:",
    round(sales_final["NetMerchandiseRevenue"].sum(), 2)
)

print()
print("Processed file exists:", processed_file.exists())

=== DATA PREPARATION FINAL CHECK ===
Rows: 1028761
Columns: 35
Exact duplicates: 0
Missing descriptions: 0
Missing dates: 0
Missing countries: 0

Gross Merchandise Sales: 19645617.81
Return Value: 716532.13
Net Merchandise Revenue: 18929085.68

Processed file exists: True
